# 06 · Evaluate and decide (GPU session)

Writes `results/killtest_v1/eval/metrics.json` and `decision.md`, then the figures. Copy the whole `results/killtest_v1/` folder back into the repo and commit it.

In [ ]:
# --- Setup: get the code (no project logic in notebooks; everything lives in src/ and scripts/) ---
import os
import subprocess
import sys

REPO_URL = "https://github.com/powershell1/Almanac.git"
BRANCH = "master"
SRC = "/tmp/Almanac"
if os.path.exists("/kaggle/input/almanac-src/pyproject.toml"):
    SRC = "/kaggle/input/almanac-src"          # offline option: repo uploaded as a Dataset
elif not os.path.exists(SRC):
    url = REPO_URL
    try:                                       # private repo: add a Kaggle Secret GITHUB_TOKEN
        from kaggle_secrets import UserSecretsClient
        tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
        url = REPO_URL.replace("https://", f"https://{tok}@")
    except Exception:
        pass
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, SRC], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "zuko==1.6.0"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", SRC], check=True)
os.chdir(SRC)
print(subprocess.run(["git", "log", "-1", "--format=%H %s"], capture_output=True, text=True).stdout)
WORK = "/kaggle/working"
CFG = "configs/killtest.yaml"


In [ ]:
TEST = "/kaggle/input/gdr4-03-simulate/shards/test"
REF = "/kaggle/input/gdr4-04-reference/shards/reference"
TRAINED = "/kaggle/input/gdr4-05-train"
subprocess.run(["cp", "-rn", f"{TRAINED}/checkpoints", f"{TRAINED}/results", WORK], check=True)

In [ ]:
!python scripts/evaluate.py --config {CFG} --root {WORK} --set paths.test_dir={TEST} paths.reference_dir={REF}

In [ ]:
!python scripts/figures/plot_killtest.py --config {CFG} --root {WORK}